In [1]:
import sqlite3
import pandas as pd

# Connect to Chinook database
conn = sqlite3.connect('chinook.db')

print("Connected to Chinook database")

Connected to Chinook database


In [2]:
# Configure pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

In [5]:
SCHEMA REFERENCE
Key Tables andColumns
tracks

TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer
Milliseconds, Bytes, UnitPrice

albums

AlbumId, Title, ArtistId

artists

ArtistId, Name

customers

CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId

invoices

InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total

invoice_items

InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity

genres

GenreId, Name

media_types

MediaTypeId, Name

employees

EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email

SyntaxError: invalid syntax (1543158634.py, line 1)

In [5]:
# Problem 1: Rank customers by total spending

query = """
        SELECT 
            Customer.FirstName AS Name,
            SUM(Invoice.Total) AS TotalSpending,
            RANK() OVER (ORDER BY SUM(Invoice.Total) DESC) AS SpendingRank
        FROM Customer
        JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
        GROUP BY Customer.CustomerId, Customer.FirstName;
        """

result = pd.read_sql_query(query, conn)
print(result)

         Name  TotalSpending  SpendingRank
0      Helena          49.62             1
1     Richard          47.62             2
2        Luis          46.62             3
3    Ladislav          45.62             4
4        Hugh          45.62             4
5       Frank          43.62             6
6       Julia          43.62             6
7        Fynn          43.62             6
8      Astrid          42.62             9
9      Victor          42.62             9
10      Terhi          41.62            11
11  František          40.62            12
12   Isabelle          40.62            12
13   Johannes          40.62            12
14       Luís          39.62            15
15   François          39.62            15
16      Bjørn          39.62            15
17       Jack          39.62            15
18        Dan          39.62            15
19    Heather          39.62            15
20       João          39.62            15
21      Wyatt          39.62            15
22   Jennif

In [7]:
# Problem 2: Rank Customers within each country

query = """
            SELECT 
                Customer.FirstName AS Customer,
                SUM(Invoice.Total) AS Total_Spending,
                Customer.Country,
                RANK() OVER (PARTITION BY Customer.Country ORDER BY SUM(Invoice.Total) DESC) AS CountryRank
            FROM Customer
            JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
            GROUP BY Customer.CustomerId, Customer.FirstName, Customer.Country;
        
        """

result = pd.read_sql_query(query, conn)
print(result)

     Customer  Total_Spending         Country  CountryRank
0       Diego           37.62       Argentina            1
1        Mark           37.62       Australia            1
2      Astrid           42.62         Austria            1
3        Daan           37.62         Belgium            1
4        Luís           39.62          Brazil            1
5     Eduardo           37.62          Brazil            2
6   Alexandre           37.62          Brazil            2
7     Roberto           37.62          Brazil            2
8    Fernanda           37.62          Brazil            2
9    François           39.62          Canada            1
10   Jennifer           38.62          Canada            2
11       Mark           37.62          Canada            3
12     Robert           37.62          Canada            3
13     Edward           37.62          Canada            3
14     Martha           37.62          Canada            3
15      Aaron           37.62          Canada           

In [16]:
# Problem 3: Running total of revenue over time

query = """
        SELECT
            Invoice.InvoiceDate,
            SUM(Invoice.Total) AS DailyRevenue,
            SUM(SUM(Invoice.Total)) OVER(ORDER BY Invoice.InvoiceDate) AS Running_Total
        FROM Invoice
        GROUP BY Invoice.InvoiceDate;
        
            
        
        """

result = pd.read_sql_query(query, conn)
print(result)

             InvoiceDate  DailyRevenue  Running_Total
0    2021-01-01 00:00:00          1.98           1.98
1    2021-01-02 00:00:00          3.96           5.94
2    2021-01-03 00:00:00          5.94          11.88
3    2021-01-06 00:00:00          8.91          20.79
4    2021-01-11 00:00:00         13.86          34.65
..                   ...           ...            ...
349  2025-12-05 00:00:00          3.96        2297.90
350  2025-12-06 00:00:00          5.94        2303.84
351  2025-12-09 00:00:00          8.91        2312.75
352  2025-12-14 00:00:00         13.86        2326.61
353  2025-12-22 00:00:00          1.99        2328.60

[354 rows x 3 columns]


In [22]:
# Problem 4: Compare each invoice amount to the customer's average invoice

query = """
        SELECT Customer.FirstName AS Customer,
            Invoice.Total AS Invoice_Amount,
            AVG(Invoice.Total) OVER(PARTITION BY Customer.CustomerId) as CustomerAvg,
            Invoice.Total - AVG(Invoice.Total) OVER(PARTITION BY Customer.CustomerId) AS Difference
        FROM Customer
        JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId;
        
        
        
        
            
        
        """

result = pd.read_sql_query(query, conn)
print(result)

    Customer  Invoice_Amount  CustomerAvg  Difference
0       Luís            3.98     5.660000   -1.680000
1       Luís            3.96     5.660000   -1.700000
2       Luís            5.94     5.660000    0.280000
3       Luís            0.99     5.660000   -4.670000
4       Luís            1.98     5.660000   -3.680000
..       ...             ...          ...         ...
407     Puja            5.94     6.106667   -0.166667
408     Puja            1.99     6.106667   -4.116667
409     Puja            1.98     6.106667   -4.126667
410     Puja           13.86     6.106667    7.753333
411     Puja            8.91     6.106667    2.803333

[412 rows x 4 columns]
